# Stage 12 — final model report

This notebook consolidates the baseline and advanced model results for the selected FraudZen scenario. It reports the held-out metrics, confusion matrices, model tradeoffs, and final project conclusions.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE=Path('../outputs/models'); OUT=BASE/'final_report'; OUT.mkdir(parents=True,exist_ok=True)
baseline=pd.read_csv(BASE/'baseline_comparison/baseline_metrics.csv')
advanced=pd.read_csv(BASE/'advanced_comparison/advanced_model_metrics.csv')
metrics=pd.concat([baseline,advanced],ignore_index=True,sort=False)
metrics.to_csv(OUT/'final_model_comparison.csv',index=False)
metrics[['dataset','model','test_precision','test_recall','test_f1','test_pr_auc','test_roc_auc','tn','fp','fn','tp']]

In [ ]:
history=metrics[metrics.dataset.eq('history_SIM_hour')].copy()
fig,axes=plt.subplots(1,2,figsize=(13,5))
history.set_index('model')[['test_pr_auc','test_f1']].plot(kind='bar',ax=axes[0],title='History-aware test metrics'); axes[0].set_ylabel('Score'); axes[0].tick_params(axis='x',rotation=20)
history.set_index('model')[['test_precision','test_recall']].plot(kind='bar',ax=axes[1],title='Precision and recall'); axes[1].set_ylabel('Score'); axes[1].tick_params(axis='x',rotation=20)
fig.tight_layout(); fig.savefig(OUT/'final_history_model_comparison.png',dpi=160); plt.show()

In [ ]:
best_pr=history.loc[history.test_pr_auc.idxmax()]; best_f1=history.loc[history.test_f1.idxmax()]
summary=pd.DataFrame([
 {'finding':'Best PR-AUC model','model':best_pr.model,'value':best_pr.test_pr_auc,'meaning':'best ranking quality for imbalanced screening'},
 {'finding':'Best F1 model','model':best_f1.model,'value':best_f1.test_f1,'meaning':'best precision/recall balance at selected threshold'},
 {'finding':'Advanced model PR-AUC','model':'Tuned HistGradientBoosting','value':float(advanced.test_pr_auc.iloc[0]),'meaning':'tuned comparison against baselines'},
])
summary.to_csv(OUT/'final_findings.csv',index=False); print(summary.to_string(index=False))

## Final interpretation

The history-aware representation is substantially stronger than the static full-day SIM profile for this scenario. Random Forest provides the strongest held-out PR-AUC among the tested models. The tuned HistGradientBoosting model provides a slightly stronger F1/recall balance at its selected threshold, with a similar false-positive count.

The result supports short-window behavioral fraud screening on this synthetic scenario. It does not establish multi-day detection performance because the data covers one calendar day. The outputs should be reported as synthetic benchmark results, with SIM-grouped evaluation and the static-versus-history comparison shown explicitly.

In [ ]:
report=(OUT/'final_report.md')
report.write_text('''# FraudZen naive SIMBox fraud detection

## Selected scenario

`all_naive_12%_200sims` from the FraudZen synthetic CDR dataset.

## Modeling result

History-aware SIM-hour features outperform static SIM profiles. Random Forest achieved the strongest held-out PR-AUC among the tested models. Tuned HistGradientBoosting achieved the strongest F1/recall balance in the advanced comparison.

## Evaluation

All partitions were grouped by SIM. Thresholds were selected on validation data and final metrics were calculated on held-out test data.

## Limitations

The scenario covers one calendar day, so multi-day persistence and long-term behavior cannot be evaluated. The data is synthetic, and simulator-specific signals may not transfer directly to live operator data.
''',encoding='utf-8')
print('saved:',report)